In [ ]:
!nvidia-smi

In [ ]:
# ============================================================
#                INSTALL REQUIRED PACKAGE
# ============================================================

!pip install -q ultralytics

from ultralytics import YOLO
import os
import shutil
from pathlib import Path

In [ ]:
import gdown

In [ ]:
!gdown "https://drive.google.com/uc?export=download&id=1XpWWyCfb5Bhdza44vVFs2qOFX8AGDaJP"

In [ ]:

import os
ZIP_FILE = "/content/Spacenet.zip"
EXTRACT_DIR = "/content/dataset"

if os.path.exists(ZIP_FILE):
    !unzip -q "{ZIP_FILE}" -d "{EXTRACT_DIR}"
    print("Dataset extracted to:", EXTRACT_DIR)
else:
    print("Zip file not found. Skip this cell if using Drive.")


In [ ]:
# Setting the training labels path:

DATA_YAML_PATH = "/content/dataset/Spacenet/data.yaml"

In [ ]:
MODEL_NAME = "yolov8m.pt"   # Example choices: yolov8n.pt, yolov8s.pt, yolov8m.pt

model = YOLO(MODEL_NAME)
print("Selected model:", MODEL_NAME)

In [ ]:
# ============================================================
#                  SET HYPERPARAMETERS
# ============================================================


PROJECT_NAME = "yolo_competition"
RUN_NAME = "trial_01"

EPOCHS = 100
IMAGE_SIZE = 768
BATCH_SIZE = 16
OPTIMIZER = "SGD"        # We can Try: SGD, Adam, AdamW, auto
LR0 = 0.01
LRF = 0.1
MOMENTUM = 0.937
WEIGHT_DECAY = 0.0005
PATIENCE = 20

# Augmentation-related hyperparameters
HSV_H = 0.015
HSV_S = 0.6
HSV_V = 0.4
DEGREES = 5.0
TRANSLATE = 0.1
SCALE = 0.5
SHEAR = 0.0
FLIPLR = 0.5
FLIPUD = 0.0
MOSAIC = 0.7
MIXUP = 0.1

print("Run name:", RUN_NAME)

In [ ]:
# ============================================================
#                   TRAINING THE MODEL
# ============================================================

results = model.train(
    data=DATA_YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
    degrees=DEGREES,
    translate=TRANSLATE,
    scale=SCALE,
    shear=SHEAR,
    fliplr=FLIPLR,
    flipud=FLIPUD,
    mosaic=MOSAIC,
    mixup=MIXUP,
    project=PROJECT_NAME,
    name=RUN_NAME,
    pretrained=True,
    verbose=True
)

In [ ]:
# ============================================================
#                VALIDATE THE TRAINED MODEL
# ============================================================

best_model_path = "/content/runs/detect/yolo_competition/trial_01/weights/best.pt"
print("Best model path:", best_model_path)

best_model = YOLO(best_model_path)
metrics = best_model.val(data=DATA_YAML_PATH)

print(metrics)

In [ ]:
# ============================================================
#             RUN INFERENCE ON SAMPLE IMAGES
# ============================================================

SAMPLE_IMAGE_DIR = "/content/dataset/Spacenet/test/images"

if os.path.exists(SAMPLE_IMAGE_DIR):
    prediction_results = best_model.predict(
        source=SAMPLE_IMAGE_DIR,
        save=True,
        conf=0.25
    )
    print("Inference complete. Prediction images saved.")
else:
    print("Sample image folder not found. Update SAMPLE_IMAGE_DIR before running.")

In [ ]:
# ============================================================
#         DISPLAYING TRAINING FOLDER CONTENTS
# ============================================================

!find "/content/runs/detect/yolo_competition/trial_014" -maxdepth 3 -type f | sort

In [ ]:
# ============================================================
#        COPYING BEST MODEL TO A CLEAR LOCATION
# ============================================================

FINAL_MODEL_PATH = "/content/best_submission_model.pt"

if os.path.exists(best_model_path):
    shutil.copy(best_model_path, FINAL_MODEL_PATH)
    print("Copied best model to:", FINAL_MODEL_PATH)
else:
    print("best.pt not found. Make sure training completed successfully.")

In [ ]:
# ============================================================
#             SAVING HYPERPARAMETER RECORD
# ============================================================

hyperparameter_log = f'''
Model: {MODEL_NAME}
Project name: {PROJECT_NAME}
Run name: {RUN_NAME}

epochs: {EPOCHS}
imgsz: {IMAGE_SIZE}
batch: {BATCH_SIZE}
optimizer: {OPTIMIZER}
lr0: {LR0}
lrf: {LRF}
momentum: {MOMENTUM}
weight_decay: {WEIGHT_DECAY}
patience: {PATIENCE}

hsv_h: {HSV_H}
hsv_s: {HSV_S}
hsv_v: {HSV_V}
degrees: {DEGREES}
translate: {TRANSLATE}
scale: {SCALE}
shear: {SHEAR}
fliplr: {FLIPLR}
flipud: {FLIPUD}
mosaic: {MOSAIC}
mixup: {MIXUP}
'''

with open("/content/hyperparameters_used.txt", "w") as f:
    f.write(hyperparameter_log)

print("Saved hyperparameter record to /content/hyperparameters_used.txt")
print(hyperparameter_log)

In [ ]:
# ============================================================
#                 DOWNLOAD SUBMISSION FILES
# ============================================================

from google.colab import files

download_items = ["/content/best_submission_model.pt", "/content/hyperparameters_used.txt"]

for item in download_items:
    if os.path.exists(item):
        files.download(item)
    else:
        print("File not found:", item)

In [ ]:
results = model.predict(
    source=SAMPLE_IMAGE_DIR,
    conf=0.25,
    save=True,
    show_conf=False
)